In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, timedelta

In [2]:
def daterange(start_date, end_date, step_days):
    """Yield (start, end) tuples for each batch."""
    current = start_date
    while current < end_date:
        batch_end = min(current + timedelta(days=step_days-1), end_date)
        yield current, batch_end
        current = batch_end + timedelta(days=1)

def fetch_twelvedata_batch(symbol, start, end, interval="1h", max_pages=50):
    """Fetch all pages for a given date batch."""
    all_rows = []
    for page in range(1, max_pages+1):
        url = (
            f"https://twelvedata.com/markets/543571/stock/nasdaq/{symbol.lower()}/historical-data"
            f"?start_date={start.strftime('%Y-%m-%d')}"
            f"&end_date={end.strftime('%Y-%m-%d')}"
            f"&interval={interval}&page={page}"
        )
        print(f"Fetching: {url}")
        resp = requests.get(url)
        soup = BeautifulSoup(resp.text, "html.parser")
        table = soup.find("table")
        if not table:
            print("No table found, stopping.")
            break
        rows = table.find("tbody").find_all("tr")
        if not rows:
            print("No more rows, stopping.")
            break
        for row in rows:
            cols = [td.get_text(strip=True) for td in row.find_all("td")]
            all_rows.append(cols)
        # Stop if less than 50 rows (last page)
        if len(rows) < 50:
            break
    return all_rows

# Main scraping loop
symbol = "nvda"
start_date = datetime(2019, 1, 7)
end_date = datetime(2024, 9, 30)
batch_days = 30  # 1 month per batch

all_data = []
for batch_start, batch_end in daterange(start_date, end_date, batch_days):
    batch_rows = fetch_twelvedata_batch(symbol, batch_start, batch_end)
    all_data.extend(batch_rows)

Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-01-07&end_date=2019-02-05&interval=1h&page=1
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-01-07&end_date=2019-02-05&interval=1h&page=2
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-02-06&end_date=2019-03-07&interval=1h&page=1
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-02-06&end_date=2019-03-07&interval=1h&page=2
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-03-08&end_date=2019-04-06&interval=1h&page=1
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-03-08&end_date=2019-04-06&interval=1h&page=2
Fetching: https://twelvedata.com/markets/543571/stock/nasdaq/nvda/historical-data?start_date=2019-04-07&end_date=2019-05-06&interval=1h&page=1

In [3]:
# Organize into DataFrame
columns = ["Date", "Open", "High", "Low", "Close", "% Change", "Volume"]
df = pd.DataFrame(all_data, columns=columns)
df = df.dropna(subset=["Date"])  # Remove empty rows
df["Date"] = pd.to_datetime(df["Date"])
for col in ["Open", "High", "Low", "Close"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["Volume"] = pd.to_numeric(df["Volume"].str.replace(",", ""), errors="coerce")

In [4]:
# Save to CSV
df = df.sort_values("Date")
df.to_csv(f"{symbol}_hourly_{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}_with_vol.csv", index=False)
print(f"Saved to {symbol}_hourly_{start_date.strftime('%Y-%m-%d')}_{end_date.strftime('%Y-%m-%d')}_with_vol.csv")

Saved to nvda_hourly_2019-01-07_2024-09-30_with_vol.csv
